# EXP-006 — video-to-audio on free Kaggle

Adds contextual audio to an **existing** Wan2.1 clip. Nothing here touches
Supabase, the worker, or any Cretona secret.

**Before Run All:**

1. Settings -> Accelerator -> **GPU T4** (one is enough)
2. Settings -> Internet -> **On**
3. Add Input -> upload one short mp4 as a private dataset

Cells are ordered cheapest-first. Cell 1 can end the experiment before a
single weight is downloaded.

> Checkpoints are **CC-BY-NC 4.0**: coursework yes, commercial use no.


## Cell 1 — kill criterion 1: does the preinstalled torch qualify?

MMAudio needs PyTorch >= 2.5.1. **Do not upgrade torch to get past this.**
The Wan2.1 worker depends on the CUDA build Kaggle ships, and breaking
that to chase audio would trade a working video path for an unproven one.


In [ ]:
import torch, sys, platform

print('python :', platform.python_version())
print('torch  :', torch.__version__)
print('cuda   :', torch.version.cuda, '| available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu    :', p.name, f'{p.total_memory/1e9:.1f} GB')

major, minor, *rest = torch.__version__.split('+')[0].split('.')
ok = (int(major), int(minor), int(rest[0]) if rest and rest[0].isdigit() else 0) >= (2, 5, 1)
print()
print('QUALIFIES:' , ok)
if not ok:
    print('KILL CRITERION 1 MET. Record the negative result in README.md and stop.')
    print('Do NOT pip install a newer torch: that is what the Wan2.1 worker runs on.')


## Cell 2 — find the input clip

Resolves whatever mp4 you attached, without assuming a dataset mount path.


In [ ]:
from pathlib import Path

clips = sorted(Path('/kaggle/input').rglob('*.mp4'))
assert clips, 'No mp4 under /kaggle/input. Attach a dataset containing one clip.'
for c in clips:
    print(f'{c.stat().st_size:>10,} B  {c}')

CLIP = clips[0]
print()
print('using:', CLIP)


## Cell 3 — install MMAudio

Installed without touching torch. If pip tries to pull a different torch,
stop: that is kill criterion 1 arriving late.


In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/hkchengrex/MMAudio.git
%cd /kaggle/working/MMAudio
!python -m pip install -q -e . 2>&1 | tail -5

import torch
print('torch after install:', torch.__version__)


## Cell 4 — generate (revised 2026-09-03)

Three changes after the first run, mirroring what `kaggle_worker.py` now does:

1. **The prompt is filtered.** The job's prompt was written for a video model
   and carries style and camera clauses. Asking MMAudio to sonify "natural
   depth of field" wastes the conditioning; setting words like "rainy" and
   "cityscape" are kept because they genuinely describe sound.
2. **A negative prompt.** On a quiet four-second clip MMAudio tends to invent a
   music bed or a voice, and both are instantly recognisable as wrong.
3. **VRAM is sampled during the run, not after.** The first run reported
   `0.00 GB` because `torch.cuda.max_memory_allocated()` reads this process,
   while `demo.py` has its own CUDA context in a subprocess.


In [ ]:
import re, subprocess, sys, threading, time
from pathlib import Path

# Same marker list as backend/kaggle_worker.py. Keep the two in step.
VISUAL_ONLY = (
    'camera ', 'in vibrant anime', 'in cyberpunk', 'in 3d animation',
    'in vintage 35mm', 'in 8k hyperrealistic', 'cinematic 16:9', 'octane render',
    'depth of field', 'lens blur', 'color grading', 'lighting', 'aesthetic',
    'hand-drawn', 'detailed textures', 'imax', 'grain',
    'subsurface scattering', 'character rendering', 'hyperrealistic',
)

def audio_prompt(video_prompt):
    clauses = [c.strip() for c in video_prompt.split(',')]
    kept = [c for c in clauses if c and not any(m in c.lower() for m in VISUAL_ONLY)]
    return ', '.join(kept) if kept else video_prompt.strip()

NEGATIVE = 'music, soundtrack, speech, voice, narration, singing, static, hum'
VIDEO_PROMPT = 'A red paper boat on a calm lake at sunrise, in 8k hyperrealistic cinematic look, ultra detailed textures, IMAX camera quality, natural depth of field'
DURATION = 4

PROMPT = audio_prompt(VIDEO_PROMPT)
print('video prompt :', VIDEO_PROMPT)
print('audio prompt :', PROMPT)
print()

# Sample the card while the subprocess runs; the parent's torch counters cannot
# see it and an after-the-fact reading shows an idle GPU.
peak = {'mib': 0}
stop = threading.Event()

def sample():
    while not stop.is_set():
        try:
            out = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=5,
            ).stdout
            peak['mib'] = max(peak['mib'], max(int(v) for v in out.split()))
        except Exception:
            pass
        time.sleep(1)

watcher = threading.Thread(target=sample, daemon=True)
watcher.start()
started = time.time()

!python demo.py --duration={DURATION} --video='{CLIP}' --prompt='{PROMPT}' --negative_prompt='{NEGATIVE}' --num_steps=25 --cfg_strength=4.5

elapsed = time.time() - started
stop.set(); watcher.join(timeout=3)

print()
print(f'wall clock : {elapsed:.0f} s      (kill criterion 3: > 300 s)')
print(f'peak VRAM  : {peak["mib"]/1024:.2f} GB  (kill criterion 2: > 13 GB)')


## Cell 5 — what came out


In [ ]:
from pathlib import Path
import IPython.display as ipd

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

outputs = sorted(Path('/kaggle/working/MMAudio/output').rglob('*'))
for o in outputs:
    if o.is_file():
        print(f'{o.stat().st_size:>10,} B  {o.name}')

videos = [o for o in outputs if o.suffix == '.mp4']
if videos:
    display(ipd.Video(str(videos[0]), embed=True, width=480))


## Cell 6 — record the verdict

Fill this in and paste it into `README.md`. A negative result recorded is
worth more than an experiment quietly abandoned.


In [ ]:
verdict = {
    'date': '2026-09-__',
    'torch_version': '',
    'qualified': None,          # cell 1
    'seconds_per_clip': None,   # cell 4
    'peak_vram_gb': None,       # cell 4 / nvidia-smi
    'audio_matches_video': None,  # your own ears, kill criterion 4
    'decision': '',             # 'proceed to integration' | 'rejected: <reason>'
}
for k, v in verdict.items():
    print(f'{k:22}: {v}')
